# Fine-tune Whisper-medium (Bengali) — Kaggle T4

Thin driver over the tested `shono.train` package: a **full** fine-tune with an
8-bit AdamW optimizer (the config that fits a free T4), scored during training
with the frozen `shono.eval` normalizer — never Whisper's.

**Resumable (mandatory):** re-running after a killed/pre-empted session continues
from the latest *complete* checkpoint automatically. Nothing is lost to the 12 h cap.

In [ ]:
# torch ships with the Kaggle image; install the rest of the training stack.
!pip install -q 'transformers>=4.44' 'accelerate>=0.33' 'bitsandbytes>=0.43' 'datasets>=2.20' 'librosa>=0.10' 'evaluate>=0.4'
# Install the shono package from the repo checkout (adjust the path to your clone).
!pip install -q -e /kaggle/working/shono

In [ ]:
import os
from pathlib import Path

from shono.data import LicenseRegistry, Manifest
from shono.provenance import RunContext, seed_everything
from shono.train import (
    TrainConfig, build_examples, render_experiment, run_training, smoke_step, total_hours,
)

# Set your Hugging Face token via Kaggle secrets (gated models / Hub uploads).
os.environ.setdefault('HF_TOKEN', '')

REPO = Path('/kaggle/working/shono')      # the shono repo checkout
DATA = REPO / 'data'
AUDIO_ROOT = Path('/kaggle/input')        # where the manifests' audio paths resolve
OUTPUT_DIR = '/kaggle/working/whisper-medium-bn-ft'

# REQUIRED: the base checkpoint to fine-tune. The project's base is a Bengali
# whisper-medium fine-tune (full-FT, 8-bit AdamW). Confirm the exact HF repo id
# on huggingface.co before running; 'openai/whisper-medium' is the generic fallback.
MODEL_ID = ''

SEED = 42
seed_everything(SEED)

## Data — build the training mix, license-checked

Manifests are generated from the licensed sources on this machine (audio never
leaves it). `validate_against` enforces the license floor: no excluded source, and
no eval-only source used as training data. `build_examples` additionally refuses any
`test` split — the last guard against train/test leakage.

In [ ]:
registry = LicenseRegistry.load(DATA / 'licenses.json')

train_manifests = [Manifest.from_jsonl(DATA / p) for p in [
    'slr53-train.manifest.jsonl',
    'cv-bn-train.manifest.jsonl',
    'bengali-loop-train.manifest.jsonl',
    'mucs-cs-train.manifest.jsonl',
]]
eval_manifests = [Manifest.from_jsonl(DATA / 'cv-bn-dev.manifest.jsonl')]
for m in [*train_manifests, *eval_manifests]:
    m.validate_against(registry)

cfg = TrainConfig(model_id=MODEL_ID, output_dir=OUTPUT_DIR, seed=SEED)
train_examples = build_examples(
    train_manifests, chunk_length_s=cfg.chunk_length_s,
    min_chunk_length_s=cfg.min_chunk_length_s,
    timestamp_sample_fraction=cfg.timestamp_sample_fraction, seed=cfg.seed,
)
eval_examples = build_examples(
    eval_manifests, chunk_length_s=cfg.chunk_length_s,
    min_chunk_length_s=cfg.min_chunk_length_s, timestamp_sample_fraction=0.0, seed=cfg.seed,
)
print(f'{len(train_examples)} train examples, {total_hours(train_examples):.1f} h')

## Smoke, then train

The smoke step runs one CPU training step on a tiny model (seconds) to catch a
broken loop before committing hours of GPU. Then the real run — resumable.

In [ ]:
# Fast sanity check of the training loop before the multi-hour run.
print(smoke_step('/kaggle/working/_smoke'))

In [ ]:
metrics = run_training(cfg, train_examples, eval_examples, audio_root=AUDIO_ROOT)
print(metrics)

## Record the experiment

Every run records its seed, full config, environment, and metrics. Fill the eval
numbers from the full evaluation (`python -m shono.eval --report m3`) once inference
over the held-out slices is done.

In [ ]:
ctx = RunContext.capture(
    cfg.seed, cfg.to_dict(),
    command='notebooks/train_whisper_medium.ipynb', repo=str(REPO),
)
record = render_experiment(
    name='whisper-medium-bn-ft-v1',
    hypothesis='whisper-medium full fine-tune beats zero-shot large-v3 and the base checkpoint',
    config=cfg, run_context=ctx,
    data_sources=sorted({m.source for m in train_manifests}),
    n_examples=len(train_examples), train_hours=total_hours(train_examples),
    eval_command='python -m shono.eval --report m3',
    metrics={k: metrics.get(k) for k in ('eval_wer', 'eval_cer') if metrics.get(k) is not None},
)
print(record)
Path(OUTPUT_DIR, 'experiment.md').write_text(record, encoding='utf-8')